# 02 — Adapt Models & Fine-tuning

**Lab 2 — Group 4 | CIFAR-10**

Bước 4: thay classifier, đóng băng backbone, mở các lớp cuối. Dùng trực tiếp `src/models.py` của nhóm.

Cấu hình được đặt trực tiếp trong cell code đầu tiên của notebook. `review` đọc kết quả lịch sử; `full` chạy mới. Mỗi notebook có thể mở bằng kernel riêng.

In [1]:
from pathlib import Path
import sys, os
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'configs').is_dir() and (p/'src').is_dir()), None)
assert ROOT is not None, 'Hãy mở notebook trong thư mục dự án đã giải nén.'
sys.path.insert(0, str(ROOT))
# Cấu hình trực tiếp trong notebook (không cần file cấu hình riêng).
MODE = "review"  # review: kết quả gốc; full: chạy thực tế
RUN_ID = "run_01"  # notebook 04–06 phải dùng cùng tên khi chạy full
DEVICE_OVERRIDE = "auto"  # auto / cpu / cuda
CPU_THREADS = 1
LIVE = MODE == "full"
assert MODE in ("review", "full")
DOWNLOAD = True
import io, json, random, hashlib, time, gc
from pathlib import Path
from datetime import datetime
from uuid import uuid4

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 42
CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
           "dog", "frog", "horse", "ship", "truck"]
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

DATA_DIR = ROOT / 'data'
SESSION_DIR = ROOT / 'outputs' / 'notebook_runs' / RUN_ID
FIG_DIR = SESSION_DIR / 'figures'
SAVE_FIGS = LIVE

def save_fig(name):
    if SAVE_FIGS:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        plt.savefig(FIG_DIR / name, dpi=150, bbox_inches="tight")

random.seed(SEED)
np.random.seed(SEED)
if LIVE:
    import torch
    import torchvision
    from torch import nn
    from torch.utils.data import DataLoader, Subset, random_split
    from torchvision import transforms, models
    from PIL import Image
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    from torch.utils.tensorboard import SummaryWriter
    import torch.nn.functional as F

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

    set_seed(SEED)
    DEVICE = torch.device(("cuda" if torch.cuda.is_available() else "cpu")
                          if DEVICE_OVERRIDE == "auto" else DEVICE_OVERRIDE)
    if DEVICE.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA không khả dụng. Chọn CPU hoặc sửa môi trường.")
    if DEVICE.type == "cpu":
        torch.set_num_threads(CPU_THREADS)
    print("PyTorch:", torch.__version__, "| Torchvision:", torchvision.__version__)
    print("Device:", DEVICE)
    print("Data:", DATA_DIR, "| Output mới:", SESSION_DIR)
else:
    print("Review: không cần import PyTorch, không tạo file training.")

Review: không cần import PyTorch, không tạo file training.


## 4. Adapt the Model for a Specific Task — Điều chỉnh cho CIFAR-10
Thay classifier cuối từ 1.000 đầu ra thành 10 đầu ra. Hai chiến lược theo Lab 2:
- **Classifier-only:** freeze toàn bộ backbone, mở toàn bộ classifier. VGG16 có nhiều lớp fully connected trong classifier.
- **Fine-tuning:** ngoài classifier, mở thêm `features.denseblock4` và `features.norm5` của DenseNet121.

Đầu ra là logits `[batch_size, 10]`, không dùng Softmax trước CrossEntropyLoss.

### Cài đặt trong module gốc
`build_model` tải weights, thay lớp cuối, đặt `requires_grad` và thống kê tham số. VGG16 classifier-only huấn luyện toàn bộ classifier, không chỉ lớp Linear cuối.

In [2]:
from IPython.display import Code
path = ROOT/'src'/'models.py'
display(Code(path.read_text(encoding='utf-8'), language='python'))

from __future__ import annotations

from typing import Any

import torch
from torch import nn
from torchvision import models

_SUPPORTED_MODELS = {"vgg16", "densenet121"}
_SUPPORTED_STRATEGIES = {"classifier_only", "finetune_last_layers"}

def _get_pretrained_weights(model_name: str, pretrained: bool):
    """Return the appropriate Torchvision weights enum or None."""
    if not pretrained:
        return None
    if model_name == "vgg16":
        return models.VGG16_Weights.DEFAULT
    if model_name == "densenet121":
        return models.DenseNet121_Weights.DEFAULT
    raise ValueError(f"Unsupported model_name: {model_name!r}")

def _replace_classifier(model: nn.Module, model_name: str, num_classes: int) -> None:
    """Replace the model's final classification layer in-place."""
    if model_name == "vgg16":
        in_features = model.classifier[-1].in_features
        model.classifier[-1] = nn.Linear(in_features, num_classes)
    elif model_name == "densenet121":
        in_features = model.classifier.in_features
        model.classifier = nn.Linear(in_features, num_classes)
    else:  # defensive; model_name được xác thực trong build_model
        raise ValueError(f"Unsupported model_name: {model_name!r}")

def _configure_trainable_parameters(
    model: nn.Module,
    model_name: str,
    strategy: str,
    finetune_layers: list[str] | tuple[str, ...] | None,
) -> list[str]:
    """Freeze the backbone and enable the classifier and requested layers.

    Returns the names of explicitly selected fine-tuning prefixes. Prefixes are
    validated against actual parameter names so a typo cannot silently freeze
    the entire backbone.
    """
    for parameter in model.parameters():
        parameter.requires_grad = False

    # Bộ phân loại luôn có thể huấn luyện được cho cả hai chiến lược được hỗ trợ.
    classifier = model.classifier
    for parameter in classifier.parameters():
        parameter.requires_grad = True

    if strategy == "classifier_only":
        return []

    if not finetune_layers:
        raise ValueError(
            "strategy='finetune_last_layers' requires a non-empty "
            "config['finetune_layers'] list of module-name prefixes."
        )

    named_parameters = list(model.named_parameters())
    valid_names = [name for name, _ in named_parameters]
    unmatched = [
        prefix for prefix in finetune_layers
        if not any(name == prefix or name.startswith(prefix + ".") for name in valid_names)
    ]
    if unmatched:
        examples = ", ".join(valid_names[:12])
        raise ValueError(
            f"Unknown finetune_layers prefix(es): {unmatched}. "
            f"Check names for {model_name}; sample parameter names: {examples}"
        )

    #Giữ cho bộ phân loại có thể huấn luyện được và chọn lọc bỏ đóng băng các phần xương sống được yêu cầu.
    for name, parameter in named_parameters:
        if any(name == prefix or name.startswith(prefix + ".") for prefix in finetune_layers):
            parameter.requires_grad = True

    return list(finetune_layers)

def get_model_summary(model: nn.Module) -> dict[str, int]:
    """Return total and trainable parameter counts."""
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    return {"total_parameters": total, "trainable_parameters": trainable}

def set_frozen_batchnorm_eval(model: nn.Module) -> None:
    """Keep BatchNorm modules with no trainable parameters in eval mode.

    Call this immediately after ``model.train()`` in every training epoch.
    ``requires_grad=False`` freezes affine parameters but does not freeze
    BatchNorm running_mean/running_var buffers by itself. BatchNorm modules
    whose parameters are intentionally unfrozen for fine-tuning remain in train mode.
    """
    for module in model.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            par

In [3]:
import yaml
ids = ['E1_vgg16','E2_densenet121','E5_cpu_1epoch']
rows=[]
if LIVE:
    from src.models import build_model, get_model_summary
    for name in ids:
        cfg = yaml.safe_load((ROOT/'configs'/f'{name}.yaml').read_text(encoding='utf-8-sig'))
        model = build_model(cfg).eval()
        with torch.no_grad():
            logits = model(torch.zeros(1,3,224,224))
        assert logits.shape == (1,10)
        rows.append({'run':name, **get_model_summary(model), 'logits':str(tuple(logits.shape))})
        del model, logits
        gc.collect()
else:
    rows = [dict(run='E1_vgg16',total_parameters=134301514,trainable_parameters=119586826),dict(run='E2_densenet121',total_parameters=6964106,trainable_parameters=10250),dict(run='E5_cpu_1epoch',total_parameters=6964106,trainable_parameters=2170378)]
    print('Số liệu lưu trong báo cáo 02_model.md; không thực hiện forward mới.')
display(pd.DataFrame(rows))

Số liệu lưu trong báo cáo 02_model.md; không thực hiện forward mới.


,run,total_parameters,trainable_parameters
0,E1_vgg16,134301514,119586826
1,E2_densenet121,6964106,10250
2,E5_cpu_1epoch,6964106,2170378


**Kết luận:** các mô hình đều được thiết kế trả 10 logits; CrossEntropyLoss nhận logits thô, không thêm softmax. Tiếp theo chuẩn bị dữ liệu.